# 47 Piezoelectric Efficiency

How higher piezoelectric efficiency shifts the bandwidth / drive-strength design space.

**Builds on:** `41_fast_pulse_bandwidth.ipynb`

## Central question

> How does higher piezoelectric efficiency shift the bandwidth / drive-strength design space?

## Working statement

Fast-pulse operation needs fewer IDT fingers for bandwidth, while useful mechanical drive needs strong piezoelectric coupling.

Higher-efficiency platforms recover drive strength while preserving bandwidth.

The seminar outlook identifies lithium niobate on diamond as a practical lever:

\[
\text{LiNbO}_3 \text{ on diamond}
\rightarrow
\text{larger effective bandwidth / stronger piezoelectric response}
\]


## 1. Setup

Notebook 41 identified the device-level tradeoff:

\[
\text{fewer IDT fingers}
\rightarrow
\text{higher bandwidth}
\]

and

\[
\text{fewer IDT fingers}
\rightarrow
\text{lower conversion efficiency}
\]

Notebook 47 asks how higher piezoelectric efficiency shifts that design space.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)

plt.rcParams["figure.dpi"] = 120


## 2. Platform comparison

Use a relative piezoelectric-efficiency scale.

The seminar outlook highlighted lithium niobate on diamond as a higher-efficiency direction compared with AlN-on-diamond devices.

Here we use a simple relative multiplier:

\[
\eta_{AlN} = 1
\]

\[
\eta_{LiNbO_3} \approx 3
\]

to represent the slide's approximately threefold improvement.


In [ ]:
platforms = pd.DataFrame(
    [
        {
            "Platform": "AlN on diamond",
            "Relative efficiency": 1.0,
            "Interpretation": "reference platform",
        },
        {
            "Platform": "LiNbO3 on diamond",
            "Relative efficiency": 3.0,
            "Interpretation": "higher-efficiency fast-pulse platform",
        },
    ]
)

platforms


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.bar(platforms["Platform"], platforms["Relative efficiency"])

ax.set_title("Relative Piezoelectric Efficiency")
ax.set_ylabel("Relative efficiency multiplier")
ax.set_ylim(0, 3.5)

for i, value in enumerate(platforms["Relative efficiency"]):
    ax.text(i, value + 0.08, f"{value:.1f}×", ha="center", fontsize=11)

ax.grid(True, axis="y", alpha=0.3)

output = FIGURES / "47_platform_comparison.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 3. Finger-count recovery model

Let:

\[
N
\]

be the normalized IDT finger count.

Use schematic relationships:

\[
B(N) \propto \frac{1}{N}
\]

\[
D(N, \eta) \propto \eta N
\]

where:

- `B` is bandwidth
- `D` is drive strength
- `η` is relative piezoelectric efficiency

Higher `η` allows smaller `N` while maintaining useful drive strength.


In [ ]:
N = np.linspace(0.2, 1.0, 500)

def bandwidth(N):
    raw = 1 / N
    return raw / raw.max()

def drive_strength(N, eta):
    raw = eta * N
    return raw / 3.0  # normalize to LiNbO3 at N=1

B = bandwidth(N)
D_aln = drive_strength(N, eta=1.0)
D_linbo3 = drive_strength(N, eta=3.0)

drive_threshold = 0.35
bandwidth_threshold = 0.45

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(N, B, linewidth=2.5, label="Bandwidth")
ax.plot(N, D_aln, linewidth=2.5, label="Drive strength: AlN on diamond")
ax.plot(N, D_linbo3, linewidth=2.5, label="Drive strength: LiNbO3 on diamond")

ax.axhline(drive_threshold, linestyle="--", linewidth=1, label="drive threshold")
ax.axhline(bandwidth_threshold, linestyle=":", linewidth=1, label="bandwidth threshold")

ax.set_title("Higher Piezoelectric Efficiency Recovers Drive Strength")
ax.set_xlabel("Normalized IDT finger count")
ax.set_ylabel("Normalized response")
ax.grid(True, alpha=0.3)
ax.legend()

output = FIGURES / "47_efficiency_multiplier.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 4. Minimum finger count for useful drive

For a required drive strength:

\[
D_{target}
\]

the required finger count scales as:

\[
N_{required} \propto \frac{D_{target}}{\eta}
\]

Higher piezoelectric efficiency shifts the required finger count lower.

Lower finger count supports higher bandwidth.


In [ ]:
eta_values = np.linspace(0.8, 4.0, 300)
D_target = 0.45

N_required = D_target / eta_values
N_required = np.clip(N_required, 0.2, 1.0)

B_available = bandwidth(N_required)

fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(eta_values, N_required, linewidth=2.5, label="Required finger count")
ax.set_title("Efficiency Multiplier Reduces Required Finger Count")
ax.set_xlabel("Relative piezoelectric efficiency")
ax.set_ylabel("Required normalized IDT finger count")
ax.grid(True, alpha=0.3)
ax.legend()

output = FIGURES / "47_required_finger_count.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(eta_values, B_available, linewidth=2.5)

ax.set_title("Efficiency Multiplier Increases Available Bandwidth")
ax.set_xlabel("Relative piezoelectric efficiency")
ax.set_ylabel("Available normalized bandwidth")
ax.grid(True, alpha=0.3)

output = FIGURES / "47_available_bandwidth.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 5. Fast-pulse design map

The design map uses two axes:

\[
N = \text{normalized IDT finger count}
\]

\[
\eta = \text{relative piezoelectric efficiency}
\]

A fast-pulse-ready region has:

- sufficient bandwidth
- sufficient drive strength

The favorable region is low `N` and high `η`.


In [ ]:
finger_count = np.linspace(0.2, 1.0, 400)
piezo_eff = np.linspace(0.5, 4.0, 400)

F, ETA = np.meshgrid(finger_count, piezo_eff)

B_map = 1 / F
B_map = B_map / B_map.max()

D_map = ETA * F
D_map = D_map / D_map.max()

bandwidth_ready = B_map >= bandwidth_threshold
drive_ready = D_map >= drive_threshold
fast_pulse_ready = bandwidth_ready & drive_ready

# Score for display: rewards simultaneous bandwidth and drive.
score = B_map * D_map

fig, ax = plt.subplots(figsize=(8, 6))

im = ax.imshow(
    score,
    origin="lower",
    aspect="auto",
    extent=[finger_count.min(), finger_count.max(), piezo_eff.min(), piezo_eff.max()],
)

ax.contour(
    F,
    ETA,
    bandwidth_ready.astype(float),
    levels=[0.5],
    colors="white",
    linewidths=2.0,
)

ax.contour(
    F,
    ETA,
    drive_ready.astype(float),
    levels=[0.5],
    colors="black",
    linewidths=2.0,
)

ax.scatter([0.8], [1.0], s=90, label="reference platform")
ax.scatter([0.4], [3.0], s=90, label="higher-efficiency region")

ax.text(0.34, 3.35, "fast-pulse-ready\nregion", fontsize=10, ha="center")
ax.text(0.78, 1.25, "reference\nregion", fontsize=10, ha="center")

ax.set_title("Fast-Pulse Design Map")
ax.set_xlabel("Normalized IDT finger count")
ax.set_ylabel("Relative piezoelectric efficiency")
ax.legend(loc="upper right")

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Bandwidth × drive response")

output = FIGURES / "47_fast_pulse_design_map.png"
plt.savefig(output, dpi=300, bbox_inches="tight")
plt.show()

print(f"saved: {output}")


## 6. Region classification

Classify the design map into three practical regimes:

1. bandwidth-ready
2. drive-ready
3. fast-pulse-ready

The fast-pulse-ready region satisfies both bandwidth and drive requirements.


In [ ]:
def classify_region(N, eta):
    b = bandwidth(np.array([N]))[0]
    d = drive_strength(np.array([N]), eta=eta)[0]
    if b >= bandwidth_threshold and d >= drive_threshold:
        return "fast-pulse-ready"
    if b >= bandwidth_threshold:
        return "bandwidth-ready, drive-seeking"
    if d >= drive_threshold:
        return "drive-ready, bandwidth-seeking"
    return "needs bandwidth and drive"

samples = pd.DataFrame(
    [
        {
            "Design": "reference platform",
            "N": 0.8,
            "eta": 1.0,
        },
        {
            "Design": "reduced-finger reference",
            "N": 0.4,
            "eta": 1.0,
        },
        {
            "Design": "higher-efficiency reduced-finger",
            "N": 0.4,
            "eta": 3.0,
        },
        {
            "Design": "high-efficiency medium-finger",
            "N": 0.6,
            "eta": 3.0,
        },
    ]
)

samples["Region"] = [classify_region(row.N, row.eta) for row in samples.itertuples()]
samples["Normalized bandwidth"] = [bandwidth(np.array([row.N]))[0] for row in samples.itertuples()]
samples["Normalized drive strength"] = [drive_strength(np.array([row.N]), row.eta)[0] for row in samples.itertuples()]

samples


## 7. Interpretation

Piezoelectric efficiency acts as the device-level lever that lets bandwidth improve while preserving useful mechanical drive strength.

The seminar roadmap can be written as:

\[
\text{reduce IDT fingers}
\rightarrow
\text{increase bandwidth}
\rightarrow
\text{recover drive through higher } \eta
\rightarrow
\text{fast-pulse-ready platform}
\]

This identifies higher-efficiency materials platforms as a practical bridge between the dressed-state regime of Notebook 37 and cavity-compatible operation.


## 8. Next notebook

Notebook 53 can combine the full sequence:

\[
\text{coherence protection}
\rightarrow
\text{dressed-state splitting}
\rightarrow
\text{fast-pulse bandwidth}
\rightarrow
\text{piezoelectric efficiency}
\rightarrow
\text{cavity-compatible operation}
\]

Candidate title:

> `53_cavity_compatible_operation.ipynb`

Candidate question:

> How do coherence protection, pulse bandwidth, and piezoelectric efficiency combine into cavity-compatible operation?
